In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import ssl
import urllib.request
ssl._create_default_https_context = ssl._create_unverified_context

# 1. Set up Hyperparameters and Device
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 3
# Automatically use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Download and Load the MNIST Dataset
# Transforms convert the raw images to PyTorch Tensors and normalize the pixel values (0 to 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Define the Convolutional Neural Network (CNN)
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        # First Convolutional Layer: 1 input channel (grayscale), 16 output channels/filters
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second Convolutional Layer: 16 input channels, 32 output channels
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected (Dense) Layers
        # After two MaxPools, the 28x28 image is reduced to 7x7. 
        # Flattened size: 32 channels * 7 * 7 = 1568
        self.fc1 = nn.Linear(in_features=32 * 7 * 7, out_features=128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(in_features=128, out_features=10) # 10 outputs (digits 0-9)

    def forward(self, x):
        # Extract features
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        
        # Flatten the 2D feature maps into a 1D vector
        x = x.view(-1, 32 * 7 * 7) 
        
        # Classify the features
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

model = MNIST_CNN().to(device)

# 4. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training Loop
print("\nStarting Training...")
for epoch in range(EPOCHS):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Zero out the gradients from the previous step
        optimizer.zero_grad()
        
        # Forward pass: make predictions
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass: calculate gradients
        loss.backward()
        
        # Update model weights
        optimizer.step()
        
        running_loss += loss.item()
        
        if (i+1) % 300 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

# 6. Evaluation (Testing) Loop
print("\nStarting Testing...")
model.eval() # Set model to evaluation mode (disables dropout/batchnorm updates)
correct = 0
total = 0

# Disable gradient calculation for testing to save memory and speed up computation
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        
        # Get the index of the highest probability class
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy on the 10,000 test images: {accuracy:.2f}%")

Training on device: cpu


100%|██████████| 9.91M/9.91M [00:10<00:00, 987kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 46.9kB/s]
100%|██████████| 1.65M/1.65M [00:13<00:00, 122kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 293kB/s]



Starting Training...
Epoch [1/3], Step [300/938], Loss: 0.0771
Epoch [1/3], Step [600/938], Loss: 0.0642
Epoch [1/3], Step [900/938], Loss: 0.0159
Epoch [2/3], Step [300/938], Loss: 0.0093
Epoch [2/3], Step [600/938], Loss: 0.2173
Epoch [2/3], Step [900/938], Loss: 0.0293
Epoch [3/3], Step [300/938], Loss: 0.0963
Epoch [3/3], Step [600/938], Loss: 0.0139
Epoch [3/3], Step [900/938], Loss: 0.0652

Starting Testing...
Test Accuracy on the 10,000 test images: 98.90%


In [3]:
# Add this to the end of mnist_cnn.py
torch.save(model.state_dict(), 'mnist_cnn.pth')
print("Model saved to mnist_cnn.pth")

Model saved to mnist_cnn.pth
